## Dataset Base

**[Titanic - Machine Learning from Disaster (Kaggle)](https://www.kaggle.com/c/titanic)**

O objetivo é usar _Machine Learning_ para criar um modelo que faça previsões sobre quais passageiros sobreviveram ao naufrágio do Titanic.

### Dicionário de Dados
Compreender os dados é o primeiro passo antes de qualquer análise. Segundo o Kaggle, temos as seguintes variáveis:

| Variável | Definição | Chave |
|---|---|---|
| `survival` | Sobrevivência (Target) | 0 = Não, 1 = Sim |
| `pclass` | Classe da passagem | 1 = 1ª, 2 = 2ª, 3 = 3ª (Proxy para Status Socioeconômico) |
| `sex` | Sexo | |
| `Age` | Idade em anos | Fracionada se < 1. Estimada em formato `xx.5` |
| `sibsp` | Nº de irmãos/cônjuges a bordo | Cônjuge, irmão, irmã, meio-irmão/irmã |
| `parch` | Nº de pais/filhos a bordo | Pai, mãe, filho, filha, enteado(a) |
| `ticket` | Número da passagem | |
| `fare` | Tarifa do passageiro | |
| `cabin` | Número da cabine | |
| `embarked` | Porto de embarque | C = Cherbourg, Q = Queenstown, S = Southampton |

---

# PARTE 1 — Fundamentos

## Importação

In [22]:
import pandas as pd
import numpy as np

## Leitura de CSV

In [23]:
df = pd.read_csv("train.csv")

## Exploração Inicial

In [24]:
df.head()
df.tail()
df.shape
df.columns
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


---

# PARTE 2 — Manipulação Essencial

## Seleção

In [25]:
df["Age"]
df[["Age", "Fare"]]

,Age,Fare
0,22.0,7.2500
1,38.0,71.2833
2,26.0,7.9250
3,35.0,53.1000
4,35.0,8.0500
...,...,...
886,27.0,13.0000
887,19.0,30.0000
888,NaN,23.4500
889,26.0,30.0000


## Filtros

In [26]:
df[df["Age"] > 30]
df[(df["Age"] > 30) & (df["Sex"] == "female")]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S
18,19,0,3,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",female,31.0,1,0,345763,18.0000,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S
865,866,1,2,"Bystrom, Mrs. (Karolina)",female,42.0,0,0,236852,13.0000,NaN,S
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C


## Criando Features

> [!NOTE]
> De acordo com o Dicionário de Dados, `SibSp` representa irmãos/cônjuges e `Parch` representa pais/filhos. Somar ambos (+1 para o próprio passageiro) nos dá o **Tamanho da Família**, uma nova *feature* (variável) muito mais útil para o modelo.

In [27]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

## Ordenação

In [28]:
df.sort_values("Fare", ascending=False)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C,2
258,259,1,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C,1
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C,1
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,263.0000,C23 C25 C27,S,6
438,439,0,1,"Fortune, Mr. Mark",male,64.0,1,4,19950,263.0000,C23 C25 C27,S,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
806,807,0,1,"Andrews, Mr. Thomas Jr",male,39.0,0,0,112050,0.0000,A36,S,1
815,816,0,1,"Fry, Mr. Richard",male,NaN,0,0,112058,0.0000,B102,S,1
466,467,0,2,"Campbell, Mr. William",male,NaN,0,0,239853,0.0000,NaN,S,1
481,482,0,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,NaN,S,1


## Valores Nulos

In [29]:
df.isnull().sum()
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

---

# PARTE 3 — GroupBy e Análise

In [30]:
df.groupby("Sex")["Survived"].mean()
df.groupby(["Sex", "Pclass"])["Survived"].mean()

pd.pivot_table(df, values="Survived", index="Sex", columns="Pclass")

Pclass,1,2,3
Sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


---

# PARTE 4 — Transformações

## Apply

In [31]:
df["AgeGroup"] = df["Age"].apply(lambda x: "Child" if x < 12 else "Adult")

## Encoding

In [32]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

## Drop

> [!NOTE]
> Estamos descartando temporariamente o `Name`, `Ticket` e `Cabin` para simplificar nosso primeiro modelo. No entanto, o `Name` contém pronomes de tratamento (Mr., Mrs., Master) e o `Cabin` tem letras que indicam o convés do navio. Em projetos mais avançados, extraímos essas sub-informações em vez de apagá-las simplesmente.

In [33]:
df.drop(["Name", "Ticket", "Cabin"], axis=1, inplace=True)

---

# PARTE 5 — Pipeline Profissional

In [34]:
df = pd.read_csv("train.csv")

# Tratar Nulos
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Feature Engineering
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Encoding
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

# Features Descartáveis (por enquanto) e Separação X/y
cols_to_drop = ["Survived", "PassengerId", "Name", "Ticket", "Cabin"]
X = df.drop(columns=cols_to_drop, errors="ignore")
y = df["Survived"]

---

# PARTE 6 — Integração com Machine Learning

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

model.score(X_test, y_test)

0.7821229050279329

---

# PARTE 7 — Técnicas Avançadas

## Query

In [36]:
df.query("Age > 30 and Sex == 1")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,FamilySize,IsAlone,Embarked_Q,Embarked_S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,2,0,False,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,2,0,False,True
11,12,1,1,"Bonnell, Miss. Elizabeth",1,58.0,0,0,113783,26.5500,C103,1,1,False,True
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",1,55.0,0,0,248706,16.0000,NaN,1,1,False,True
18,19,0,3,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",1,31.0,1,0,345763,18.0000,NaN,2,0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",1,48.0,0,0,17466,25.9292,D17,1,1,False,True
865,866,1,2,"Bystrom, Mrs. (Karolina)",1,42.0,0,0,236852,13.0000,NaN,1,1,False,True
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",1,47.0,1,1,11751,52.5542,D35,3,0,False,True
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",1,56.0,0,1,11767,83.1583,C50,2,0,False,False


## Assign

In [37]:
df = (
    df
    .assign(FamilySize=lambda x: x.SibSp + x.Parch + 1)
)

## Value Counts Normalizado

In [38]:
df["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

## Category Type

In [39]:
df["Pclass"] = df["Pclass"].astype("category")

## Uso de Memória

In [40]:
df.memory_usage(deep=True)

Index            132
PassengerId     7128
Survived        7128
Pclass           915
Name           67685
Sex             7128
Age             7128
SibSp           7128
Parch           7128
Ticket         49674
Fare            7128
Cabin          32712
FamilySize      7128
IsAlone         7128
Embarked_Q       891
Embarked_S       891
dtype: int64

---

# PARTE 8 — Submissão para o Kaggle

Para avaliar seu modelo oficialmente no Kaggle, temos que ler o arquivo `test.csv`, submetê-lo às **mesmas transformações do pipeline** e gerar o arquivo de submissão (ex: `submission.csv`). O objetivo final do Kaggle é prever a coluna "Survived" que *não* existe no arquivo de teste!

In [41]:
# 1. Carregar os dados de teste
df_test = pd.read_csv("test.csv")
passageiros_id = df_test["PassengerId"] # Salvar para a submissão final

# 2. Aplicar Pipeline de Tratamento (Mesmas lógicas do treino)
df_test["Age"] = df_test["Age"].fillna(df["Age"].median())
df_test["Fare"] = df_test["Fare"].fillna(df["Fare"].median()) # Teste possui 1 Fare nulo
df_test["Embarked"] = df_test["Embarked"].fillna("S")

df_test["FamilySize"] = df_test["SibSp"] + df_test["Parch"] + 1
df_test["IsAlone"] = (df_test["FamilySize"] == 1).astype(int)

df_test["Sex"] = df_test["Sex"].map({"male": 0, "female": 1})
df_test = pd.get_dummies(df_test, columns=["Embarked"], drop_first=True)

# 3. Remover Colunas Inúteis para o Modelo
X_pred = df_test.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")

# 4. Realizar a Previsão no modelo treinado
previsoes = model.predict(X_pred)

# 5. Montar Arquivo de Submissão final
submission = pd.DataFrame({
    "PassengerId": passageiros_id,
    "Survived": previsoes
})
submission.to_csv("minha_primeira_submissao.csv", index=False)

> [!TIP]
> Agora basta ir ao Kaggle, procurar pelo botão **Submit Predictions** e enviar o arquivo `minha_primeira_submissao.csv` gerado na sua pasta. Veja sua posição no ranking!

---

# Roadmap de Evolução

- **Semana 1**: Manipulação básica e filtros
- **Semana 2**: Feature engineering e encoding
- **Semana 3**: Pipeline e otimização
- **Semana 4**: Novos datasets e desafios Kaggle

---

# Conclusão

Domine:

- Seleção avançada
- GroupBy complexo
- Feature engineering
- Encoding
- Pipeline profissional

Você estará pronto para aplicar Pandas em projetos reais de Machine Learning.